# Homework 04: Model deployment

Author: Rongshan Wei 

UNI: rw3082

Date: April 24th, 2026

## Project Overview
This notebook marks the transition from model experimentation to Model Deployment within the Machine Learning (ML) lifecycle. Using the Formula 1 (F1) historical dataset, the primary objective is to build and compare two distinct predictive models—Random Forest (RF) and Linear Regression (LR). Unlike previous exercises, this assignment emphasizes the operationalization of ML outputs by logging models via MLflow and programmatically writing batch predictions into dedicated SQL database tables for downstream consumption.

## Technical Stack
* Platform: Databricks (Cloud-based Spark environment)
* Storage & Database: Unity Catalog / Hive Metastore (SQL-based table management)
* ML Engines: RandomForestRegressor (Ensemble Learning) & Linear Regression (Statistical Baseline)
* Experiment Tracking: MLflow (Logging parameters, models, 4+ metrics, and artifacts)
* Version Control: GitHub (Final code submission and versioning)

## Dataset Description
The analysis continues to leverage the integrated F1 dataset sourced from AWS S3, focusing on the relational links between:
* drivers & results: To establish the lationship between driver attributes and finishing positions.
* races & constructors: To provide temporal context and technical performance features.

## Methodology & Best Practices
To adhere to industrial MLOps standards and the specific requirements of Homework #5, the workflow is structured as follows:
* Database Initialization: Creating two distinct SQL tables in a personal schema to serve as the destination for model predictions.
* Comparative Model Building: * Developing two separate ML pipelines (RF and LR).
  * Utilizing MLflow to log a minimum of four metrics (e.g., MSE, RMSE, MAE, R2) and two artifacts per model.
* Batch Inference & Persistence: * Executing predictions on the test dataset for each model.
  * Converting predictions into Spark DataFrames and persisting them into the pre-defined database tables.
* Selection & Analysis: Utilizing the MLflow Tracking UI to evaluate model performance and justify the deployment of results.

## I. Global Configuration & Environment
Before processing large-scale datasets, it is a Best Practice to document the computational environment to ensure reproducibility.
* Spark Version: 3.x (Databricks Runtime)
* Cluster Configuration: Serverless Compute / Standard Runtime
* ML Engine: Scikit-learn (RF and Linear Regression)
* Tracking Server: Databricks Managed MLflow
* Primary Language: Python 3.10+

## II. Library Ingestion

1. Logic Formulation

Following PEP 8 guidelines, all imports are centralized at the top of the notebook. In this iteration, I have expanded the library stack to include:

* Model Diversity: Added LinearRegression to compare against RandomForestRegressor.
* Persistence Logic: Included pyspark.sql components to handle the conversion of predictions from Pandas to Spark for database ingestion.
* Enhanced Metrics: Added logic to log four (4) distinct metrics (MSE, RMSE, MAE, R2) as required by the rubric.
* Environment Stability: Explicitly included statsmodels to support advanced residual visualizations.

2. Implementation

In [0]:
# --- Standard Library Imports ---
import os
import sys

# --- Third-party Library Imports: Data Processing ---
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

# --- Machine Learning & Visualization ---
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels
import tempfile

# Models
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

# Metrics
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## III. Data Ingestion & Feature Engineering

1. Logic Formulation
To build a high-quality predictive model, I will aggregate multiple relational tables from the F1 dataset to create a comprehensive feature set.
The Strategy:
* Core Table: Use results as the primary dataset containing our target variable (positionOrder).
* Contextual Joins: Join with races to capture temporal/circuit factors, drivers for biographical features, and constructors to account for team technical superiority.
* Feature Selection: I will focus on attributes known to impact race outcomes: driver age, starting grid position, and historical team performance.
* Target Definition: We will prepare the data for a Regression task—predicting the final positionOrder.

In [0]:
# Configuration: AWS S3 / Databricks Volume Path
# Adjust this path based on your specific S3 mount or volume
DATA_PATH = '/Volumes/gr5069/raw/f1_data/'

# Load relevant datasets
df_results = spark.read.csv(f"{DATA_PATH}results.csv", header=True, inferSchema=True)
df_races = spark.read.csv(f"{DATA_PATH}races.csv", header=True, inferSchema=True)
df_drivers = spark.read.csv(f"{DATA_PATH}drivers.csv", header=True, inferSchema=True)
df_constructors = spark.read.csv(f"{DATA_PATH}constructors.csv", header=True, inferSchema=True)
df_status = spark.read.csv(f"{DATA_PATH}status.csv", header=True, inferSchema=True)

# Feature Engineering: Joining and selecting relevant columns
# We focus on the modern era (e.g., post-2010) for more consistent data patterns
ml_data = (
    df_results.select("raceId", "driverId", "constructorId", "grid", "positionOrder", "statusId")
    .join(df_races.select("raceId", "year", "circuitId"), on="raceId")
    .join(df_drivers.select("driverId", "dob"), on="driverId")
    .join(df_constructors.select("constructorId", "name"), on="constructorId")
    # Filter for completed races to reduce noise from random mechanical failures
    .filter(F.col("statusId") == 1) 
)

# Convert to Pandas for Scikit-learn compatibility (standard practice for medium-sized F1 data)
final_df = ml_data.toPandas()

# Preliminary Data Profile
print(f"Total records for ML training: {len(final_df)}")
display(final_df.head(10))

## IV. Database & Table Initialization

1. Logic Formulation
According to the requirements of Homework #05, model deployment involves persisting inference results into a structured SQL environment. This ensures that predictions are accessible for downstream analysis and auditing.
- Namespace Management: We define a dedicated schema (database) where the prediction tables will reside. Given the permissions in the shared Databricks environment, we utilize the assigned user-specific schema.
- Table Idempotency: We ensure the environment is "clean" by dropping any pre-existing versions of the prediction tables. This allows us to satisfy the "Create two (2) new tables" requirement during each execution.
- Scalability: By initializing these tables within the Hive Metastore, we enable other Spark or SQL sessions to query the model results without re-running the ML pipeline.

2. Implementation

In [0]:
# --- Configuration: Define Schema and Table Names ---
MY_SCHEMA = "gr5069.rw3082" 

TABLE_RF = f"{MY_SCHEMA}.predictions_rf"
TABLE_LR = f"{MY_SCHEMA}.predictions_lr"

# 1. Ensure the Database exists (if permissions allow)
# If this throws a Permission Denied error, simply skip this line 
# and use an existing schema assigned by the instructor.
try:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS `{MY_SCHEMA}`")
    print(f"Schema '{MY_SCHEMA}' is ready.")
except Exception as e:
    print(f"Note: Could not create schema (likely due to permissions). Using existing: {MY_SCHEMA}")

# 2. Drop existing tables to ensure they are 'NEW' as per assignment instructions
spark.sql(f"DROP TABLE IF EXISTS `{TABLE_RF}`")
spark.sql(f"DROP TABLE IF EXISTS `{TABLE_LR}`")

print(f"  Deployment environment initialized:")
print(f"   - Target Table A: {TABLE_RF}")
print(f"   - Target Table B: {TABLE_LR}")

## IV. Model Building & MLflow Experimentation

1. Logic Formulation

To satisfy the requirements of Homework #05, I have refactored the training pipeline to support two distinct architectures: Random Forest (RF) and Linear Regression (LR).
- Expanded Metrics: Each run now logs four metrics: Mean Squared Error (MSE), Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and R-squared (R^2).
- Artifact Consistency: Both models generate a residual plot and a feature importance/coefficient CSV file using tempfile.
- Deployment Readiness: The training function now returns the prediction array, enabling the subsequent task of writing results to SQL tables.

2. Implementation

In [0]:
# 1. Feature Engineering
final_df['dob'] = pd.to_datetime(final_df['dob'])
final_df['driver_age'] = final_df['year'] - final_df['dob'].dt.year

# Separation: Identifiers are used for subsequent database storage, while features are used for model training
identifiers = ['raceId', 'driverId']
features = ['grid', 'driver_age', 'constructorId', 'year']
target = ['positionOrder']

# Clean data
model_df = final_df.dropna(subset=features + target)

# 2. Prepare the training set and test set
X = model_df[identifiers + features] 
y = model_df[target]

X_train_full, X_test_full, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Data ultimately fed into the model (excluding identifiers)
X_train = X_train_full[features]
X_test = X_test_full[features]

print(f"Setup Complete: Training on {len(X_train)} samples with {len(features)} features.")

In [0]:
# --- MLflow Experiment Setup ---
experiment_path = f"/Users/rw3082@columbia.edu/take-home-exercise-4-WrsRosanne/HW04_Model_Deployment"

# Set up an experiment and retrieve the ID
mlflow.set_experiment(experiment_path)
experimentID = mlflow.get_experiment_by_name(experiment_path).experiment_id

print(f"   MLflow Experiment is set!")
print(f"   Path: {experiment_path}")
print(f"   ID: {experimentID}")

In [0]:
def train_and_log_model(experimentID, run_name, model_type, params, X_train, X_test, y_train, y_test, X_test_full):
    import os
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import mlflow.sklearn
    import seaborn as sns
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    import tempfile

    with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
        # --- A. Model Training ---
        if model_type == "RF":
            model = RandomForestRegressor(**params)
        else:
            model = LinearRegression(**params)
            
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        # --- B. Logging Model & Parameters ---
        mlflow.sklearn.log_model(model, f"{model_type}-model")
        [mlflow.log_param(param, value) for param, value in params.items()]

        # --- C. Logging FOUR Metrics ---
        mse = mean_squared_error(y_test, predictions)
        rmse = np.sqrt(mse) 
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)  
        mlflow.log_metric("r2", r2)  
        
        print(f"✅ {run_name} Results -> MSE: {mse:.3f}, RMSE: {rmse:.3f}, R2: {r2:.3f}")

        # --- D. Logging TWO Artifacts ---
        # 1. Feature Importance / Coefficients CSV
        if model_type == "RF":
            importance_df = pd.DataFrame(list(zip(X_train.columns, model.feature_importances_)), 
                                        columns=["Feature", "Importance"])
        else:
            importance_df = pd.DataFrame(list(zip(X_train.columns, model.coef_.flatten())), 
                                        columns=["Feature", "Coefficient"])
            
        temp_csv = tempfile.NamedTemporaryFile(prefix="feature-data-", suffix=".csv")
        try:
            importance_df.to_csv(temp_csv.name, index=False)
            mlflow.log_artifact(temp_csv.name, "model_features.csv")
        finally:
            temp_csv.close()
        
        # 2. Residual Plot
        fig, ax = plt.subplots()
        sns.residplot(x=predictions.flatten(), y=y_test.values.flatten().astype(float), lowess=False)
        plt.xlabel("Predicted Position")
        plt.ylabel("Residual")
        plt.title(f"Residual Plot: {run_name}")
        
        temp_plot = tempfile.NamedTemporaryFile(prefix="residuals-", suffix=".png")
        try:
            fig.savefig(temp_plot.name)
            mlflow.log_artifact(temp_plot.name, "residuals.png")
        finally:
            temp_plot.close()
        plt.close(fig)
        
        # --- E. Prepare Deployment DataFrame ---
        # Re-attach raceId and driverId for database persistence
        results_df = X_test_full[['raceId', 'driverId']].copy()
        results_df['actual_position'] = y_test.values
        results_df['predicted_position'] = predictions
        results_df['model_type'] = model_type
        
        return results_df

In [0]:
# --- 1. Execute Model A: Random Forest ---
print(" Training and Logging Model A: Random Forest...")

# Using the best parameters identified in HW03
rf_params = {
    "n_estimators": 100, 
    "max_depth": 10, 
    "min_samples_split": 10,
    "random_state": 42
}

# Run RF and capture the results for deployment
rf_results_pd = train_and_log_model(
    experimentID, 
    "RF_Deployment_Run", 
    "RF", 
    rf_params, 
    X_train, X_test, y_train, y_test, X_test_full
)

# --- 2. Execute Model B: Linear Regression ---
print("\n Training and Logging Model B: Linear Regression...")

lr_params = {"fit_intercept": True}

# Run LR and capture the results for deployment
lr_results_pd = train_and_log_model(
    experimentID, 
    "LR_Deployment_Run", 
    "LR", 
    lr_params, 
    X_train, X_test, y_train, y_test, X_test_full
)

# --- 3. Persistent Storage (The Deployment Step) ---
print(f"\nWriting predictions to database: {MY_SCHEMA}...")

# Write RF Predictions
rf_spark_df = spark.createDataFrame(rf_results_pd)
rf_spark_df.write.mode("overwrite").saveAsTable(f"{MY_SCHEMA}.predictions_rf")

# Write LR Predictions
lr_spark_df = spark.createDataFrame(lr_results_pd)
lr_spark_df.write.mode("overwrite").saveAsTable(f"{MY_SCHEMA}.predictions_lr")

print("Deployment Complete! Predictions stored in SQL tables.")

In [0]:
# View the prediction results of the Random Forest
print("--- RF Predictions Table ---")
display(spark.table(f"{MY_SCHEMA}.predictions_rf").limit(10))

# View the prediction results of the linear regression
print("\n--- LR Predictions Table ---")
display(spark.table(f"{MY_SCHEMA}.predictions_lr").limit(10))

## Conclusion & Model Selection

Based on the MLflow experiment results, the Random Forest model is the superior choice for deployment, achieving a significantly higher R^2 (0.589 vs 0.463) and a lower RMSE (3.31 vs 3.78) compared to the Linear Regression baseline. While Linear Regression provides a quick statistical reference, it fails to capture the complex, non-linear interactions inherent in Formula 1 racing data.

The successful persistence of predictions into the gr5069.rw3082 database marks the completion of the model deployment phase. These tables are now ready for downstream consumption, such as building dashboards to compare predicted versus actual podium finishes across different Grand Prix.